### Recipe 1. A CSV with latitude and longitude columns

In [1]:
%pip install pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import geopandas as gpd
import os

In [3]:
df_trees = pd.read_csv("../data/public_trees.csv")
df_trees = df_trees.dropna(subset=["Latitude", "Longitude"])          # count what this drops: print(len(df))
gdf_trees = gpd.GeoDataFrame(df_trees, geometry=gpd.points_from_xy(df_trees["Longitude"], df_trees["Latitude"]), crs="EPSG:4326")
gdf_trees=gdf_trees[["ARROND", "ARROND_NOM", "Essence_ang", "DHP", "Date_Plantation", "Coord_X", "Coord_Y", "Longitude", "Latitude", "geometry"]].to_file("../data/Public_Trees.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

C:\Users\ryand\AppData\Local\Temp\ipykernel_34568\97672461.py:1: DtypeWarning: Columns (4: Rue, 5: Rue_cote, 15: LOCALISATION, 16: Localisation_code, 17: CODE_PARC, 18: NOM_PARC, 19: Rue_de, 20: Rue_a, 22: Distance_ligne_rue, 23: Stationnement_jour, 24: Stationnement_heure, 28: Nom_secteur) have mixed types. Specify dtype option on import or set low_memory=False.
  df_trees = pd.read_csv("../data/public_trees.csv")


### Recipe 2. A CSV with x and y in metres (projected)

You need the system's code. Look in the portal's metadata for "projection", "CRS", "SCR" or "EPSG". Common ones: Montréal **MTM zone 8** is `EPSG:2950` (or the older `EPSG:32188`; the two differ by about a metre). Québec Lambert is `EPSG:32198`. Vancouver and much of BC: UTM zone 10N, `EPSG:26910`. Toronto: MTM zone 10, `EPSG:2952`. If you cannot find it, ask your TA before you guess.

In [4]:
# df = pd.read_csv("../data/YOUR_FILE.csv")
# gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["X_COLUMN"], df["Y_COLUMN"]), crs="EPSG:YOUR_CODE")
# gdf = gdf.to_crs("EPSG:4326")
# gdf[["FIELD_1", "FIELD_2", "geometry"]].to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 3. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [5]:
gdf_heat = gpd.read_file("../data/heat_island/ICU_20250821.shp")      # or .gpkg, or .geojson
print(gdf_heat.crs, len(gdf_heat), sorted(gdf_heat.geom_type.unique()))
print(gdf_heat.columns.tolist())
gdf_heat = gdf_heat[["TempMoyVF", "Shape_Area", "geometry"]]
# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)
# # gdf["geometry"] = gdf.geometry.simplify(5)
gdf_heat = gdf_heat.to_crs("EPSG:4326")
gdf_heat.to_file("../data/heat_island/ICU_20250821.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

EPSG:2950 556967 ['Polygon']
['gridcode', 'Type_sol', 'TempMoyVF', 'ClassesNB_', 'ClassQual', 'Shape_Leng', 'Shape_Area', 'geometry']


In [6]:
# ---- Heat by borough: average the 557k heat cells into Montréal's 19 boroughs ----
 
# 1. Borough boundaries (downloaded from Montréal's open data portal into data/)
boroughs = gpd.read_file("../data/limites-administratives-agglomeration-nad83.geojson")
print(boroughs.columns.tolist())
print(boroughs["TYPE"].value_counts())                 # boroughs vs. the island's other cities
boroughs = boroughs[boroughs["TYPE"] == "Arrondissement"][["NOM", "geometry"]]
print(len(boroughs), "boroughs")                       # should print 19
 
# 2. Heat cells, without the water cells (those have a temperature of 0)
cells = gpd.read_file("../data/heat_island/ICU_20250821.shp")
cells = cells[cells["TempMoyVF"] > 0][["TempMoyVF", "ClassesNB_", "geometry"]]
print(len(cells), "land cells")
 
# 3. Put both in the heat file's system (metres), then turn each 30 m cell into its centre point.
#    Joining points is much faster than overlapping half a million squares with 19 shapes.
boroughs = boroughs.to_crs(cells.crs)
cells["geometry"] = cells.geometry.centroid
 
# 4. Which borough is each cell in? Then summarise per borough.
joined = gpd.sjoin(cells, boroughs, predicate="within")
stats = joined.groupby("NOM").agg(
    temp_mean=("TempMoyVF", "mean"),                                   # average surface temperature, °C
    pct_heat_island=("ClassesNB_", lambda c: (c == 5).mean() * 100),   # % of cells classed "Îlot de chaleur"
).round(1).reset_index()
 
# 5. Attach the numbers to the borough shapes, lighten them, convert, write.
gdf_heat_boroughs = boroughs.merge(stats, on="NOM")
gdf_heat_boroughs["geometry"] = gdf_heat_boroughs.geometry.simplify(10)   # 10 m, in metres before converting
gdf_heat_boroughs = gdf_heat_boroughs.to_crs("EPSG:4326")
gdf_heat_boroughs.to_file("../data/heat_island/heat_by_borough.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
 
print(gdf_heat_boroughs[["NOM", "temp_mean", "pct_heat_island"]].sort_values("temp_mean", ascending=False))
print(round(os.path.getsize("../data/heat_island/heat_by_borough.geojson") / 1_000_000, 2), "MB")
 

['CODEID', 'NOM', 'NOM_OFFICIEL', 'CODEMAMH', 'CODE_3C', 'NUM', 'ABREV', 'TYPE', 'COMMENT', 'DATEMODIF', 'geometry']
TYPE
Arrondissement    19
Ville liée        15
Name: count, dtype: int64
19 boroughs
547920 land cells
                                         NOM  temp_mean  pct_heat_island
11                             Saint-Léonard       32.6             24.8
6                                      Anjou       31.8             26.4
17                             Saint-Laurent       31.4             23.7
10                                   Lachine       31.4             21.8
12                             Montréal-Nord       31.3             15.5
1       Villeray-Saint-Michel-Parc-Extension       31.1             11.4
3                      Le Plateau-Mont-Royal       30.7              6.6
9              Mercier-Hochelaga-Maisonneuve       30.5             15.0
0                                    LaSalle       30.1             12.7
8                               Le Sud-Ouest      

In [9]:
# ---- Add Outremont's trees: same columns and same format as the main file, with no duplicate points ----
MAIN = "../data/public_trees.csv"
OUTREMONT = "../data/inventaire-arbre-outremont.csv"     # the file you downloaded, renamed to this
COMBINED = "../data/public_trees_combined.csv"           # main file + Outremont, used by the next cells

main = pd.read_csv(MAIN)
try:
    out = pd.read_csv(OUTREMONT, sep=None, engine="python", encoding="utf-8-sig")   # sep=None works out , or ;
except UnicodeDecodeError:
    out = pd.read_csv(OUTREMONT, sep=None, engine="python", encoding="latin-1")
print(len(out), "rows in the Outremont file. Its columns:")
print(out.columns.tolist())

def find(df, *names):
    """The first of these column names that exists in df (ignoring capitals), or None."""
    lower = {c.lower(): c for c in df.columns}
    return next((lower[n.lower()] for n in names if n.lower() in lower), None)

def num(series):
    """Numbers, also when written French-style with a comma (45,52)."""
    return pd.to_numeric(series.astype(str).str.replace(",", ".", regex=False), errors="coerce")

# Which Outremont column holds what. If one shows None below but the file has it under another
# name, type that name in here, e.g. "dhp": "DIAMETRE".
COLS = {
    "species": find(out, "Essence_ang", "essence_anglais", "nom_anglais", "english_name"),
    "latin":   find(out, "Essence_latin", "essence_latin", "nom_latin", "latin_name"),
    "dhp":     find(out, "DHP", "diametre", "diameter"),
    "date":    find(out, "Date_Plantation", "date_plantation", "plantation"),
    "lat":     find(out, "Latitude", "lat"),
    "lon":     find(out, "Longitude", "long", "lon"),
    "x":       find(out, "Coord_X", "x"),
    "y":       find(out, "Coord_Y", "y"),
}
print(COLS)

new = pd.DataFrame(index=out.index)

# 1. Location, checked against Montréal's area so wrong, swapped or MTM columns can't slip through
def in_montreal(lat, lon):
    return lat.between(45.3, 45.8) & lon.between(-74.1, -73.4)

lat = num(out[COLS["lat"]]) if COLS["lat"] else None
lon = num(out[COLS["lon"]]) if COLS["lon"] else None
if lat is not None and lon is not None and in_montreal(lon, lat).mean() > in_montreal(lat, lon).mean():
    lat, lon = lon, lat                                    # the file has latitude and longitude the other way round
if lat is None or lon is None or in_montreal(lat, lon).mean() < 0.5:
    # Not usable as latitude/longitude, so use Coord_X/Coord_Y (MTM zone 8, in metres) instead
    xcol, ycol = COLS["x"] or COLS["lon"], COLS["y"] or COLS["lat"]
    if xcol is None or ycol is None:
        raise ValueError("No location columns found: set 'lat'/'lon' or 'x'/'y' in COLS by hand.")
    x, y = num(out[xcol]), num(out[ycol])
    if (x > y).mean() > 0.5:
        x, y = y, x                                        # in MTM, Y (about 5,040,000) is bigger than X (about 300,000)
    pts = gpd.GeoSeries(gpd.points_from_xy(x, y), crs="EPSG:2950").to_crs("EPSG:4326")
    lat = pd.Series(pts.y.values, index=out.index)
    lon = pd.Series(pts.x.values, index=out.index)
new["Latitude"], new["Longitude"] = lat, lon
print("First locations (latitude, longitude):", new[["Latitude", "Longitude"]].head(3).round(5).values.tolist())

# 2. Species: use the main file's English name for the same Latin name, so the spelling is identical.
#    If there is no Latin name to match, use Outremont's own English name.
english = (main.dropna(subset=["Essence_latin", "Essence_ang"])
               .drop_duplicates("Essence_latin").set_index("Essence_latin")["Essence_ang"])
new["Essence_ang"] = out[COLS["latin"]].map(english) if COLS["latin"] else pd.NA
if COLS["species"]:
    new["Essence_ang"] = new["Essence_ang"].fillna(out[COLS["species"]])

# 3. Trunk diameter, rounded like the main file (whole cm) if the main file uses whole numbers
new["DHP"] = num(out[COLS["dhp"]]) if COLS["dhp"] else pd.NA
if (main["DHP"].dropna() % 1 == 0).all():
    new["DHP"] = new["DHP"].round(0)

# 4. Planting date, written exactly like the main file: 2015-06-12T00:00:00
if COLS["date"]:
    dates = pd.to_datetime(out[COLS["date"]], errors="coerce", dayfirst=True)
    new["Date_Plantation"] = dates.dt.strftime("%Y-%m-%dT00:00:00")
else:
    new["Date_Plantation"] = pd.NA

new["ARROND_NOM"] = "Outremont"

# 5. No duplicates: drop trees without a usable location (empty, or outside Montréal),
#    repeats inside the Outremont file, and any tree within 2 m of a tree already in the main file
start = len(new)
new = new[in_montreal(new["Latitude"], new["Longitude"])]
no_location = start - len(new)
new = new.drop_duplicates(subset=["Latitude", "Longitude", "Essence_ang"])
repeats = start - no_location - len(new)

main_xy = main.dropna(subset=["Latitude", "Longitude"])
g_new = gpd.GeoDataFrame(new, geometry=gpd.points_from_xy(new["Longitude"], new["Latitude"]), crs="EPSG:4326").to_crs("EPSG:2950")
g_main = gpd.GeoDataFrame(geometry=gpd.points_from_xy(main_xy["Longitude"], main_xy["Latitude"]), crs="EPSG:4326").to_crs("EPSG:2950")
minx, miny, maxx, maxy = g_new.total_bounds
g_main = g_main.cx[minx - 50:maxx + 50, miny - 50:maxy + 50]          # only main-file trees near Outremont
already = gpd.sjoin_nearest(g_new, g_main, max_distance=2, how="inner").index.unique()
new = new.drop(index=already)

print(f"{start} Outremont rows: {no_location} without a usable location, {repeats} repeated in the file, "
      f"{len(already)} already in the main file -> {len(new)} trees added")

# 6. The combined file: every main-file tree, plus the new Outremont trees
combined = pd.concat([main, new], ignore_index=True)
combined.to_csv(COMBINED, index=False)
print(len(main), "+", len(new), "=", len(combined), "trees in", COMBINED)
print("Missing values in the added trees:", new[["Essence_ang", "DHP", "Date_Plantation"]].isna().sum().to_dict())

C:\Users\ryand\AppData\Local\Temp\ipykernel_34568\3731674048.py:6: DtypeWarning: Columns (4: Rue, 5: Rue_cote, 15: LOCALISATION, 16: Localisation_code, 17: CODE_PARC, 18: NOM_PARC, 19: Rue_de, 20: Rue_a, 22: Distance_ligne_rue, 23: Stationnement_jour, 24: Stationnement_heure, 28: Nom_secteur) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv(MAIN)


7683 rows in the Outremont file. Its columns:
['INV_TYPE', 'EMP_NO', 'ARROND', 'ARROND_NOM', 'Rue', 'Rue_cote', 'No_civique', 'Emplacement', 'Coord X', 'Coord Y', 'SIGLE', 'Essence_latin', 'Essence_fr', 'Essence_ang', 'DHP', 'DATE_INV', 'DATE_MOD', 'LOCALISATION', 'Localisation_code', 'SECTEUR', 'CODE_PARC', 'NOM_PARC', 'Rue_de', 'Rue_a', 'Distance_pave', 'Distance_ligne_rue', 'Stationnement_jour', 'Stationnement_heure', 'District', 'Remarquable', 'Nom_secteur', 'Coord_X', 'Coord_Y', 'Latitude', 'Longitude']
{'species': 'Essence_ang', 'latin': 'Essence_latin', 'dhp': 'DHP', 'date': None, 'lat': 'Latitude', 'lon': 'Longitude', 'x': 'Coord_X', 'y': 'Coord_Y'}
First locations (latitude, longitude): [[45.51388, -73.61711], [45.51394, -73.61722], [45.514, -73.61732]]
7683 Outremont rows: 4 without a usable location, 1 repeated in the file, 0 already in the main file -> 7678 trees added
335480 + 7678 = 343158 trees in ../data/public_trees_combined.csv
Missing values in the added trees: {'Ess

In [10]:
# ---- Trees by borough: one small file per borough, so the site loads only what you click ----
import re, unicodedata
 
def file_name(name):
    """'Côte-des-Neiges–Notre-Dame-de-Grâce' -> 'cote-des-neiges-notre-dame-de-grace' (safe for a web address)"""
    plain = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "-", plain.lower()).strip("-")
 
# 1. The 19 boroughs from the heat-by-borough cell (run that cell first)
gdf_heat_boroughs = gpd.read_file("../data/heat_island/heat_by_borough.geojson")
 
# 2. The trees, rebuilt from the CSV (the Recipe 1 cell leaves gdf_trees empty after writing)
df_trees = pd.read_csv("../data/public_trees_combined.csv")
df_trees = df_trees.dropna(subset=["Latitude", "Longitude"])
gdf_trees = gpd.GeoDataFrame(df_trees, geometry=gpd.points_from_xy(df_trees["Longitude"], df_trees["Latitude"]), crs="EPSG:4326")
gdf_trees = gdf_trees[["Essence_ang", "DHP", "Date_Plantation", "geometry"]]
 
# 3. Which borough is each tree in? (by location, so the names always match the borough file)
trees_in = gpd.sjoin(gdf_trees, gdf_heat_boroughs[["NOM", "geometry"]], predicate="within")
print(len(gdf_trees), "trees ->", len(trees_in), "inside a borough")
 
# 4. One GeoJSON per borough
os.makedirs("../data/trees_by_borough", exist_ok=True)
for name, group in trees_in.groupby("NOM"):
    path = f"../data/trees_by_borough/{file_name(name)}.geojson"
    group[["Essence_ang", "DHP", "Date_Plantation", "geometry"]].to_file(path, driver="GeoJSON", COORDINATE_PRECISION=6)
    print(f"{name}: {len(group)} trees, {os.path.getsize(path) / 1_000_000:.1f} MB")
 
# 5. Add the file name and tree numbers to the borough file, so the site knows what to load
counts = trees_in.groupby("NOM").size()
area_km2 = gdf_heat_boroughs.to_crs("EPSG:2950").area / 1_000_000          # area in metres first, then km²
gdf_heat_boroughs["trees_file"] = gdf_heat_boroughs["NOM"].apply(file_name) + ".geojson"
gdf_heat_boroughs["tree_count"] = gdf_heat_boroughs["NOM"].map(counts).fillna(0).astype(int)
gdf_heat_boroughs["trees_per_km2"] = (gdf_heat_boroughs["tree_count"] / area_km2).round(0).astype(int)
 
os.remove("../data/heat_island/heat_by_borough.geojson")                    # replace the old version
gdf_heat_boroughs.to_file("../data/heat_island/heat_by_borough.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
print(gdf_heat_boroughs[["NOM", "temp_mean", "tree_count", "trees_per_km2"]].sort_values("temp_mean", ascending=False))
 

C:\Users\ryand\AppData\Local\Temp\ipykernel_34568\4264145283.py:13: DtypeWarning: Columns (4: Rue, 5: Rue_cote, 15: LOCALISATION, 16: Localisation_code, 17: CODE_PARC, 18: NOM_PARC, 19: Rue_de, 20: Rue_a, 22: Distance_ligne_rue, 23: Stationnement_jour, 24: Stationnement_heure, 28: Nom_secteur) have mixed types. Specify dtype option on import or set low_memory=False.
  df_trees = pd.read_csv("../data/public_trees_combined.csv")


343156 trees -> 342426 inside a borough
Ahuntsic-Cartierville: 36427 trees, 7.3 MB
Anjou: 6 trees, 0.0 MB
Côte-des-Neiges-Notre-Dame-de-Grâce: 29054 trees, 5.8 MB
L'Île-Bizard-Sainte-Geneviève: 22 trees, 0.0 MB
LaSalle: 40 trees, 0.0 MB
Lachine: 8702 trees, 1.7 MB
Le Plateau-Mont-Royal: 20004 trees, 4.0 MB
Le Sud-Ouest: 22660 trees, 4.6 MB
Mercier-Hochelaga-Maisonneuve: 38679 trees, 7.7 MB
Montréal-Nord: 20 trees, 0.0 MB
Outremont: 7670 trees, 1.3 MB
Pierrefonds-Roxboro: 16949 trees, 3.4 MB
Rivière-des-Prairies-Pointe-aux-Trembles: 34573 trees, 6.8 MB
Rosemont-La Petite-Patrie: 37413 trees, 7.4 MB
Saint-Laurent: 32 trees, 0.0 MB
Saint-Léonard: 23500 trees, 4.8 MB
Verdun: 19113 trees, 3.7 MB
Ville-Marie: 19790 trees, 4.0 MB
Villeray-Saint-Michel-Parc-Extension: 27772 trees, 5.6 MB
                                         NOM  temp_mean  tree_count  \
11                             Saint-Léonard       32.6       23500   
6                                      Anjou       31.8           6

In [11]:
# ---- Tree tiles: the trees split into a grid of ~1 km squares, so the site downloads only what's on screen ----
import json

TILE_W, TILE_H = 0.015, 0.01   # square size in degrees: about 1.2 km wide and 1.1 km tall in Montréal

df_trees = pd.read_csv("../data/public_trees_combined.csv")
df_trees = df_trees.dropna(subset=["Latitude", "Longitude"])

# Suggestion 2: keep only the date (2015-06-12T00:00:00 -> 2015-06-12), so 335,000 rows carry less text
df_trees["Date_Plantation"] = df_trees["Date_Plantation"].str.slice(0, 10)

# Which square is each tree in? (// rounds down, also for negative longitudes)
df_trees["tile"] = ((df_trees["Longitude"] // TILE_W).astype(int).astype(str) + "_" +
                    (df_trees["Latitude"] // TILE_H).astype(int).astype(str))

gdf_trees = gpd.GeoDataFrame(df_trees, geometry=gpd.points_from_xy(df_trees["Longitude"], df_trees["Latitude"]), crs="EPSG:4326")

# One small GeoJSON per square. Suggestion 2: 5 decimals is still about 1 m accurate.
os.makedirs("../data/tree_tiles", exist_ok=True)
for tile, group in gdf_trees.groupby("tile"):
    group[["Essence_ang", "DHP", "Date_Plantation", "geometry"]].to_file(
        f"../data/tree_tiles/{tile}.geojson", driver="GeoJSON", COORDINATE_PRECISION=5)

# A small index, so the site knows the square size and which squares have trees (no requests for empty ones)
tiles = sorted(gdf_trees["tile"].unique().tolist())
with open("../data/tree_tiles/index.json", "w") as f:
    json.dump({"tile_w": TILE_W, "tile_h": TILE_H, "tiles": tiles}, f)

sizes = [os.path.getsize(f"../data/tree_tiles/{t}.geojson") / 1_000_000 for t in tiles]
print(len(tiles), "squares; biggest", round(max(sizes), 2), "MB; average", round(sum(sizes) / len(sizes), 2), "MB")

C:\Users\ryand\AppData\Local\Temp\ipykernel_34568\2444329076.py:6: DtypeWarning: Columns (4: Rue, 5: Rue_cote, 15: LOCALISATION, 16: Localisation_code, 17: CODE_PARC, 18: NOM_PARC, 19: Rue_de, 20: Rue_a, 22: Distance_ligne_rue, 23: Stationnement_jour, 24: Stationnement_heure, 28: Nom_secteur) have mixed types. Specify dtype option on import or set low_memory=False.
  df_trees = pd.read_csv("../data/public_trees_combined.csv")


296 squares; biggest 0.99 MB; average 0.22 MB


In [12]:
# ---- Heat zones image: the ~550,000 zones coloured into one PNG, which the site shows as a picture ----
# Needs Pillow for saving the image. If you get "No module named PIL", run this once in its own cell:
#   %pip install pillow
import json
import numpy as np
from PIL import Image

HEAT_COLOURS = ['#fed976', '#feb24c', '#fd8d3c', '#f03b20', '#bd0026']   # same as app.js, coolest to hottest
PIXEL = 50   # pixel size in web-map units: about 35 m on the ground in Montréal (the zones are 30 m)

# 1. Borough outlines and the heat zones (without water, which has a temperature of 0)
boroughs = gpd.read_file("../data/heat_island/heat_by_borough.geojson")[["NOM", "geometry"]]
cells = gpd.read_file("../data/heat_island/ICU_20250821.shp")
cells = cells[cells["TempMoyVF"] > 0][["TempMoyVF", "geometry"]]
boroughs = boroughs.to_crs(cells.crs)

# 2. Keep only the zones inside the 19 boroughs (by centre point), like the borough layer
centres = cells.copy()
centres["geometry"] = cells.geometry.centroid
inside = gpd.sjoin(centres, boroughs, predicate="within").index
cells = cells.loc[inside]
centres = centres.loc[inside]

# 3. Five colour classes: the coolest fifth of zones gets class 0 ... the hottest fifth class 4
breaks = cells["TempMoyVF"].quantile([0.2, 0.4, 0.6, 0.8]).round(1).tolist()
heat_class = pd.cut(cells["TempMoyVF"], [float("-inf")] + breaks + [float("inf")], labels=False).astype(int).values
lows = [float(round(cells["TempMoyVF"].min(), 1))] + breaks
highs = breaks + [float(round(cells["TempMoyVF"].max(), 1))]
print("Class breaks (°C):", breaks)

# 4. Place each zone's centre on a pixel grid in the web map's own projection (EPSG:3857),
#    so the picture lines up exactly with the basemap
pts = centres.geometry.to_crs("EPSG:3857")
x, y = pts.x.values, pts.y.values
x0, top = x.min() - PIXEL, y.max() + PIXEL
cols = ((x - x0) // PIXEL).astype(int)
rows = ((top - y) // PIXEL).astype(int)
ncols, nrows = cols.max() + 2, rows.max() + 2

# 5. Paint the pixels: one colour per class, fully transparent where there is no zone
colours = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] + [255] for h in HEAT_COLOURS], dtype=np.uint8)
img = np.zeros((nrows, ncols, 4), dtype=np.uint8)
img[rows, cols] = colours[heat_class]
Image.fromarray(img, "RGBA").save("../data/heat_island/heat_zones.png", optimize=True)

# 6. Where the picture's corners are (latitude, longitude), plus the class ranges for the legend
corners = gpd.GeoSeries(gpd.points_from_xy([x0, x0 + ncols * PIXEL], [top - nrows * PIXEL, top]),
                        crs="EPSG:3857").to_crs("EPSG:4326")
info = {"bounds": [[corners.y.iloc[0], corners.x.iloc[0]], [corners.y.iloc[1], corners.x.iloc[1]]],
        "lows": lows, "highs": highs}
with open("../data/heat_island/heat_zones.json", "w") as f:
    json.dump(info, f)

print(ncols, "x", nrows, "pixels,", round(os.path.getsize("../data/heat_island/heat_zones.png") / 1_000_000, 2), "MB")

Class breaks (°C): [25.7, 28.3, 30.3, 33.1]
1051 x 921 pixels, 0.15 MB


In [ ]:
# ---- Heat zones by borough: the 30 m zones merged by colour, one small file per borough ----
# Run after the heat-by-borough and trees-by-borough cells. Takes a few minutes (merging ~550,000 squares).
# import re, unicodedata

# def file_name(name):
    # """Same naming as the trees cell, so each borough's zone and tree files share a name."""
    # plain = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    # return re.sub(r"[^a-z0-9]+", "-", plain.lower()).strip("-")

# 1. Borough outlines and the heat zones (without water, which has a temperature of 0)
# boroughs = gpd.read_file("../data/heat_island/heat_by_borough.geojson")[["NOM", "geometry"]]
# cells = gpd.read_file("../data/heat_island/ICU_20250821.shp")
# cells = cells[cells["TempMoyVF"] > 0][["TempMoyVF", "geometry"]]
# boroughs = boroughs.to_crs(cells.crs)                      # work in metres

# 2. Which borough is each zone in? (by its centre point, like the heat-by-borough cell)
# centres = cells.copy()
# centres["geometry"] = cells.geometry.centroid
# in_borough = gpd.sjoin(centres, boroughs, predicate="within")["NOM"]
# cells = cells.loc[in_borough.index].assign(NOM=in_borough)

# 3. Five colour classes: the coolest fifth of zones gets class 0 ... the hottest fifth class 4
# breaks = cells["TempMoyVF"].quantile([0.2, 0.4, 0.6, 0.8]).round(1).tolist()
# cells["heat_class"] = pd.cut(cells["TempMoyVF"], [float("-inf")] + breaks + [float("inf")], labels=False).astype(int)
# lows = [round(cells["TempMoyVF"].min(), 1)] + breaks
# highs = breaks + [round(cells["TempMoyVF"].max(), 1)]
# print("Class breaks (°C):", breaks)

# 4. Merge touching zones of the same class within each borough: ~550,000 squares become ~95 shapes
# zones = cells[["NOM", "heat_class", "geometry"]].dissolve(by=["NOM", "heat_class"], as_index=False)
# zones["geometry"] = zones.geometry.simplify(5)             # 5 m, in metres before converting
# zones["temp_from"] = zones["heat_class"].map(lambda c: lows[c])
# zones["temp_to"] = zones["heat_class"].map(lambda c: highs[c])
# zones = zones.to_crs("EPSG:4326")

# 5. One GeoJSON per borough, named like its tree file
# os.makedirs("../data/heat_zones_by_borough", exist_ok=True)
# for name, group in zones.groupby("NOM"):
    # path = f"../data/heat_zones_by_borough/{file_name(name)}.geojson"
    # group.to_file(path, driver="GeoJSON", COORDINATE_PRECISION=6)
    # print(f"{name}: {os.path.getsize(path) / 1_000_000:.1f} MB")

Class breaks (°C): [25.7, 28.3, 30.3, 33.1]
Ahuntsic-Cartierville: 0.8 MB
Anjou: 0.4 MB
Côte-des-Neiges-Notre-Dame-de-Grâce: 0.8 MB
L'Île-Bizard-Sainte-Geneviève: 0.2 MB
LaSalle: 0.6 MB
Lachine: 0.5 MB
Le Plateau-Mont-Royal: 0.3 MB
Le Sud-Ouest: 0.6 MB
Mercier-Hochelaga-Maisonneuve: 0.8 MB
Montréal-Nord: 0.4 MB
Outremont: 0.1 MB
Pierrefonds-Roxboro: 0.6 MB
Rivière-des-Prairies-Pointe-aux-Trembles: 1.3 MB
Rosemont-La Petite-Patrie: 0.5 MB
Saint-Laurent: 1.3 MB
Saint-Léonard: 0.4 MB
Verdun: 0.3 MB
Ville-Marie: 0.6 MB
Villeray-Saint-Michel-Parc-Extension: 0.6 MB


In [ ]:

# ---- settings: change these for your own data ----
FILE = "../data/heat_island/heat_by_borough.geojson"
WEST, EAST, SOUTH, NORTH = -74.0, -73.4, 45.35, 45.75   # a box around your area, in degrees
EXPECTED = 19         # how many features you expect (rows after your filters)
POPUP_FIELDS = ["NOM", "temp_mean", "pct_heat_island", "tree_count", "trees_per_km2"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed
